# AGORA Dataset Explorer\n\nExploring the new AGORA dataset (March 2026 release):\n- **documents.csv** — AI policy documents with taxonomy tags\n- **segments.csv** — Document segments with per-segment annotations\n- **authorities.csv** — Issuing authorities\n- **collections.csv** — Document collections

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

docs = pd.read_csv(DATA_DIR / "documents.csv", encoding="utf-8-sig")
segs = pd.read_csv(DATA_DIR / "segments.csv", encoding="utf-8-sig")
auths = pd.read_csv(DATA_DIR / "authorities.csv", encoding="utf-8-sig")
colls = pd.read_csv(DATA_DIR / "collections.csv", encoding="utf-8-sig")

print(f"Documents:   {len(docs):,} rows, {len(docs.columns)} columns")
print(f"Segments:    {len(segs):,} rows, {len(segs.columns)} columns")
print(f"Authorities: {len(auths):,} rows")
print(f"Collections: {len(colls):,} rows")

## Documents overview

In [ ]:
docs.head(3)

In [ ]:
# Non-taxonomy columns
meta_cols = [c for c in docs.columns if not c.startswith(("Applications:", "Harms:", "Incentives:", "Risk factors:", "Strategies:"))]
docs[meta_cols].info()

## Taxonomy tag analysis

In [ ]:
# Taxonomy columns — 77 boolean columns across 5 categories
tax_cols = [c for c in docs.columns if c.startswith(("Applications:", "Harms:", "Incentives:", "Risk factors:", "Strategies:"))]
print(f"Total taxonomy columns: {len(tax_cols)}")

# Count tags per category
categories = {}
for col in tax_cols:
    cat = col.split(":")[0]
    categories.setdefault(cat, []).append(col)

for cat, cols in categories.items():
    true_count = docs[cols].sum().sum()
    print(f"  {cat}: {len(cols)} tags, {int(true_count):,} total annotations")

In [ ]:
# Top 15 most common taxonomy tags
tag_counts = docs[tax_cols].sum().sort_values(ascending=False).head(15)
tag_counts.plot.barh(figsize=(10, 6), title="Top 15 Taxonomy Tags (document-level)")
import matplotlib.pyplot as plt
plt.xlabel("Number of documents")
plt.tight_layout()

## Authorities & Collections

In [ ]:
# Top authorities by document count
top_auths = docs["Authority"].value_counts().head(15)
print("Top 15 authorities:")
print(top_auths.to_string())

In [ ]:
# Collections
colls

## Segments overview

In [ ]:
# Segments per document distribution
segs_per_doc = segs.groupby("Document ID").size()
print(f"Segments: {len(segs):,} total across {segs_per_doc.count():,} documents")
print(f"Mean segments/doc: {segs_per_doc.mean():.1f}")
print(f"Median: {segs_per_doc.median():.0f}, Max: {segs_per_doc.max()}")

segs_per_doc.plot.hist(bins=50, figsize=(10, 4), title="Segments per document")
plt.xlabel("Number of segments")
plt.tight_layout()

In [ ]:
segs.head(3)

## Annotation coverage

In [ ]:
# Annotation and validation status
annotated = docs["Annotated?"].sum()
validated = docs["Validated?"].sum()
has_text = docs["Official plaintext retrieved"].notna().sum()
has_summary = docs["Short summary"].notna().sum() - (docs["Short summary"] == "").sum()

print(f"Annotated:    {annotated:,} / {len(docs):,} ({annotated/len(docs)*100:.1f}%)")
print(f"Validated:    {validated:,} / {len(docs):,} ({validated/len(docs)*100:.1f}%)")
print(f"Has fulltext: {has_text:,} / {len(docs):,}")
print(f"Has summary:  {has_summary:,} / {len(docs):,}")

# Tags per document (density)
tags_per_doc = docs[tax_cols].sum(axis=1)
print(f"\nTags per document — mean: {tags_per_doc.mean():.1f}, median: {tags_per_doc.median():.0f}")
print(f"Documents with 0 tags: {(tags_per_doc == 0).sum():,}")